# EDA — rfqs.csv
Exploración inicial de las solicitudes de cotización antes de integrarlas con las otras dos fuentes.

In [1]:
import pandas as pd
from pathlib import Path

## Carga y forma general
Dimensiones, tipos de columna y nulos.

In [5]:
RAW_DIR = Path.cwd().parent / "data" / "raw"

df_rfq = pd.read_csv(RAW_DIR / "rfqs.csv")
print(df_rfq.head())
print("-" * 50)
print(df_rfq.info())
print("-" * 50)
print(df_rfq.shape)
print("-" * 50)

       rfq_id             product_type underlyings basket_type  \
0  RFQ-000000      Kessel Run Snowball    CLNE|DRC    worst_of   
1  RFQ-000001      Kessel Run Snowball   SITH|WOOK    worst_of   
2  RFQ-000002  Death Star Phoenix Note   MNDO|NABO    worst_of   
3  RFQ-000003      Kessel Run Snowball   TECH|SITH    worst_of   
4  RFQ-000004     Mandalorian Twin-Win        JEDI      single   

   autocall_barrier_pct  protection_barrier_pct  no_call_period_months  \
0                1.0000                  0.5424                      3   
1                1.0000                  0.6258                      6   
2                1.0152                  0.5899                      2   
3                1.0000                  0.5944                      4   
4                1.3149                  0.7880                      0   

  observation_frequency  quoted_implied_vol  notional_credits  \
0               Monthly              0.2455            250000   
1                    1Y     

## Coherencia de fechas en RFQs no ejecutadas
Si executed es False, start_date debería coincidir con requested_date (el producto nunca llegó a emitirse).

In [6]:
mask = df_rfq["executed"] == False
print((df_rfq.loc[mask, "start_date"] == df_rfq.loc[mask, "requested_date"]).all())

True


## Coherencia target / ejecución
avg_duration_months solo debería faltar cuando executed es False.

In [7]:
print((df_rfq["avg_duration_months"].isna() == ~df_rfq["executed"]).all())

True


## Frecuencias de observación
Valores distintos de observation_frequency presentes en los datos.

In [8]:
print(df_rfq["observation_frequency"].value_counts())

observation_frequency
1M            6174
3M            5298
6M            3618
2M            1995
1Y            1601
1D             711
Monthly        670
mensual        667
1 month        658
M              651
Quarterly      576
trimestral     570
Q              566
3 months       559
12M            191
Y              175
Annual         164
anual          156
Name: count, dtype: int64


## Subyacentes de la cesta
Separa la columna underlyings (string separado por "|") en una lista de tickers.

In [9]:
df_rfq["underlyings"] = df_rfq["underlyings"].str.split("|")

## Unicidad de rfq_id
Confirma que cada solicitud tiene un identificador único.

In [11]:
print(df_rfq["rfq_id"].duplicated().sum())

0


## Distribución de product_type
Frecuencia de cada tipo de producto solicitado.

In [12]:
print(df_rfq["product_type"].value_counts())

product_type
Holocron Reverse Convertible    4245
Mandalorian Twin-Win            4216
Wretched Hive Digital           4167
Sith Eternal Snowball           4160
Death Star Phoenix Note         4148
Kessel Run Snowball             4064
Name: count, dtype: int64


## Tamaño de la cesta por basket_type
single debería tener siempre 1 subyacente; worst_of puede tener más de uno.

In [13]:
print(df_rfq.groupby("basket_type")["underlyings"].apply(lambda s: s.str.len().unique()))

basket_type
single         [1]
worst_of    [2, 3]
Name: underlyings, dtype: object


## Estadísticos descriptivos
Rangos de barreras, volatilidad cotizada, nominal y período de no-call.

In [14]:
print(df_rfq[["autocall_barrier_pct", "protection_barrier_pct", "quoted_implied_vol", "notional_credits", "no_call_period_months"]].describe())

       autocall_barrier_pct  protection_barrier_pct  quoted_implied_vol  \
count          25000.000000            25000.000000        25000.000000   
mean               1.139004                0.663538            0.278661   
std                0.147257                0.096473            0.089055   
min                0.950000                0.500000            0.093600   
25%                1.000000                0.600000            0.217700   
50%                1.152700                0.622500            0.274250   
75%                1.276100                0.760900            0.330500   
max                1.400000                0.850000            0.599700   

       notional_credits  no_call_period_months  
count      2.500000e+04           25000.000000  
mean       6.764120e+05               1.480160  
std        1.695565e+06               2.016732  
min        1.000000e+05               0.000000  
25%        1.000000e+05               0.000000  
50%        2.500000e+05       